In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers==4.35.2 torch sentencepiece -q
print("Packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.2 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
Packages installed!


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Path to shared model
model_path = "/content/drive/MyDrive/Hindi_Text_Normalizer_Model02"

# Load
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f" Model loaded on {device}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


 Model loaded on cpu


In [ ]:
import torch

# 1. Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def normalize_hindi_text(noisy_text):
    # input with the same format used in training
    input_text = noisy_text + " </s> <2hi>"
    inputs = tokenizer(input_text, return_tensors="pt", padding=True).to(device)

    # Generate with corrected length and beam search
    with torch.no_grad():
        output_tokens = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=200,
            num_beams=5,             # Better quality
            early_stopping=True,
            forced_bos_token_id=tokenizer.encode("<2hi>")[0]
        )

    # Decode and clean up
    decoded = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    return decoded.replace("<2hi>", "").strip()

# 2. Test Cases
test_samples = [
    "आज मोसम अछा है", # Spelling/Grammar noise
    "वहा धीरे धीरे चल रहा था।", # Punctuation/Dash noise
    "क्या आप बता सकते हें की स्टेशन कहा है?" # Complex question noise
]

print("="*50)
print("INDICBART HINDI NORMALIZER TEST")
print("="*50)

for noisy in test_samples:
    clean = normalize_hindi_text(noisy)
    print(f"NOISY: {noisy}")
    print(f"CLEAN: {clean}")
    print("-" * 30)

INDICBART HINDI NORMALIZER TEST
NOISY: आज मोसम अछा है
CLEAN: आज मोसम अछा है।
------------------------------
NOISY: वहा धीरे धीरे चल रहा था।
CLEAN: वहा धीरे-धीरे चल रहा था।
------------------------------
NOISY: क्या आप बता सकते हें की स्टेशन कहा है?
CLEAN: क्या आप बता सकते हैं की स्टेशन कहाँ है?
------------------------------


In [ ]:
!pip install SpeechRecognition pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 35.9 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import speech_recognition as sr
from pydub import AudioSegment
import os
import uuid

# FUNCTION 1: Manual Text Normalization
def manual_text_normalize(noisy_text):
    if not noisy_text.strip():
        return "Please enter some text."
    # Calling inference function
    return normalize_hindi_text(noisy_text)

# FUNCTION 2: Audio Pipeline
def process_audio_pipeline(audio_path):
    if audio_path is None or not os.path.exists(audio_path):
        return "Audio signal not detected.", ""
#to process audio data
    recognizer = sr.Recognizer()
    unique_wav = f"temp_{uuid.uuid4().hex}.wav"

    try:
        audio = AudioSegment.from_file(audio_path)
        audio.export(unique_wav, format="wav") #loaded audio saved on disk

        with sr.AudioFile(unique_wav) as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio_data = recognizer.record(source)
            raw_text = recognizer.recognize_google(audio_data, language="hi-IN")

        if raw_text.strip():
            # Using model function
            clean_text = normalize_hindi_text(raw_text)
            return raw_text, clean_text
        else:
            return "Speech not recognized.", "Please speak clearly."

    except Exception as e:
        return f"Error: {str(e)}", "Please try again."
    finally:
        if os.path.exists(unique_wav):
            os.remove(unique_wav)

# GRADIO INTERFACE
#  light theme using custom CSS and Soft theme
with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="orange")) as demo:
    gr.Markdown("Raw ASR Normalizer")

    with gr.Tabs():
        # TAB 1: Speech to Text
        with gr.TabItem("Audio Normalizer"):
            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Record/Upload")
                    btn_audio = gr.Button("Process Audio", variant="primary")
                with gr.Column():
                    raw_out = gr.Textbox(label="Raw ASR Output", lines=3)
                    clean_out = gr.Textbox(label="Corrected Text", lines=3)

            btn_audio.click(process_audio_pipeline, inputs=audio_input, outputs=[raw_out, clean_out])

        # TAB 2: Manual Text Input
        with gr.TabItem("Direct Text Input"):
            gr.Markdown("### Enter Raw ASR Text manually to clean it")
            with gr.Row():
                with gr.Column():
                    text_input = gr.Textbox(label="Input Noisy Text", placeholder="e.g., में वहा जा रहा हु...", lines=5)
                    btn_text = gr.Button("Normalize Text", variant="primary")
                with gr.Column():
                    text_output = gr.Textbox(label="Corrected Output", lines=5)

            btn_text.click(manual_text_normalize, inputs=text_input, outputs=text_output)

    gr.Markdown("---")
    gr.Markdown("Built with IndicBART & Google ASR Engine")

# Launch
demo.launch(debug=False, share=True)

/tmp/ipython-input-1069845282.py:46: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="orange")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e19cd14df1a92cf43.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
if 'demo' in locals():
    demo.close()

Closing server running on port: 7860
